# P14 · Laboratorio multiactivo y capitalización
**QuantLab | Módulo 1 · Fundamentos financieros | Lecciones 1.21 y 1.28**

**Pregunta:** ¿cómo comparar inversiones con distinta moneda, volatilidad y crecimiento compuesto?

Objetivos: convertir precios a EUR, alinear calendarios, distinguir precio y retorno total, calcular CAGR, volatilidad y drawdown, estudiar correlaciones y entender el valor temporal. Este cuaderno es un tutorial, no una recomendación de inversión.

**Ruta:** ejemplo resuelto → datos → métricas → gráficos → ejercicios resueltos (sección 8). Por defecto se ejecuta con datos sintéticos reproducibles; selecciona `live` para descargar históricos reales o `csv` para reutilizar tus datos.

Las funciones están en **`src/quantlab/p14/multi_asset.py`** y sus pruebas en **`tests/test_multi_asset.py`** (`pytest` desde la raíz del proyecto).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pathlib import Path
import inspect
import sys

try:
    import quantlab
except ImportError:  # sin `pip install -e .`: buscamos src/ subiendo desde el cuaderno
    src = next((d / "src" for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "quantlab").exists()), None)
    if src is None:
        raise
    sys.path.insert(0, str(src))
from quantlab.p14.multi_asset import (ASSETS, TICKERS, cagr, compound_return, dividend_paths, download_prices,
                                      equivalent_rate, eur_return, future_value, fx_decomposition, manual_compound,
                                      performance, prepare_weekly, present_value, split_halves, synthetic_prices,
                                      total_return)

plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True, 'grid.alpha': .2,
                    'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})
DATA_MODE = 'synthetic'  # synthetic | live | csv
CSV_PATH = 'multiasset_prices.csv'
START = '2021-01-01'
END = '2026-01-01'  # exclusivo
INITIAL_CAPITAL = 1000.0
SEED = 42
EXPORT = False

## 1. Capitalización: ejemplo resuelto
Con un tipo efectivo por periodo $r$, capital inicial $C_0$ y $n$ periodos: $FV=C_0(1+r)^n$, $PV=FV/(1+r)^n$. Para una secuencia variable: $C_T=C_0\prod(1+r_t)$. Una media aritmética de cero no implica mantener el patrimonio. El descuento no garantiza que esa rentabilidad se pueda conseguir.

In [ ]:
print(inspect.getsource(future_value))
print(inspect.getsource(compound_return))

r = np.array([.5, -.5])
print(f'Media aritmética: {r.mean():.2%}; retorno acumulado: {compound_return(r):.2%}')
print(f'Media geométrica: {(1+compound_return(r))**(1/len(r))-1:.2%}')
print(f'Valor actual de 10.000 EUR al 5% durante 5 años: {present_value(10000,.05,5):,.2f} EUR')
years = np.arange(21)
fig, ax = plt.subplots()
ax.plot(years, 1000*(1+.10*years), label='Interés simple')
ax.plot(years, 1000*1.10**years, label='Interés compuesto')
ax.set(title='Capital a un 10% anual constante (hipotético)', xlabel='Años', ylabel='EUR'); ax.legend(); plt.show()

## 2. Universo de inversión y fuentes
| Columna | Proxy real | Interpretación |
|---|---|---|
| BTC | BTC-USD | Precio de Bitcoin en USD |
| Acciones | SPY | ETF de renta variable estadounidense, cierre ajustado |
| Bonos | IEF | ETF de bonos del Tesoro de EE. UU., cierre ajustado |
| Oro | GLD | Vehículo de exposición al oro, cierre ajustado |
| Global | VT | ETF global, benchmark de renta variable |
| EURUSD | EURUSD=X | USD por EUR |
| USD efectivo | 1/EURUSD | Valor de un dólar en EUR, sin remuneración |

Los vehículos son ejemplos analíticos, no recomendaciones ni necesariamente accesibles para inversores minoristas europeos. Los cierres ajustados de los ETF aproximan retorno total con distribuciones reinvertidas, sin impuestos personales ni costes de negociación; no añadas otra vez los dividendos. GLD no equivale a oro físico sin costes; IEF no equivale a un bono individual mantenido hasta vencimiento.

Fuente opcional: Yahoo Finance mediante [yfinance](https://ranaroussi.github.io/yfinance/reference/yfinance.functions.html) (incluido en `requirements.txt`). No es un feed de ejecución ni point-in-time garantizado. Consulta condiciones de uso antes de guardar o redistribuir datos. Si falla la descarga, no se sustituye silenciosamente por simulaciones.

CSV: columnas `date,BTC,Acciones,Bonos,Oro,Global,EURUSD` (precios en USD y FX en USD/EUR). La fecha es la sesión de cierre; se admiten ausencias por festivos, no duplicados. Documenta proveedor, ajuste de dividendos y fechas. Para utilizar exportaciones de P04, primero agrega y convierte el BTC/EUR a USD o adapta explícitamente el esquema: no conviertas a EUR dos veces.

In [ ]:
print(inspect.getsource(download_prices))

if DATA_MODE == 'synthetic':
    raw_prices = synthetic_prices(SEED)
elif DATA_MODE == 'live':
    raw_prices = download_prices(START, END)  # si falla, lanza el error
elif DATA_MODE == 'csv':
    raw_prices = pd.read_csv(CSV_PATH, index_col='date', parse_dates=True)
else:
    raise ValueError('DATA_MODE desconocido')
print('Fuente:', DATA_MODE, '| Filas brutas:', len(raw_prices))
display(raw_prices.head(3).round(4))

## 3. Calendarios y moneda común
Compararemos niveles en EUR una vez por semana, usando la última fecha de lunes a viernes en que **todas** las series tienen observación. Se descartan las semanas de los extremos por prudencia. Si falta una semana interior, detenemos la ejecución para investigar: no tratamos un retorno de dos semanas como uno semanal.

Los cierres del mismo día no son necesariamente simultáneos (BTC opera 24/7 y el mercado de acciones no). Esta aproximación es adecuada para un ejercicio descriptivo de medio plazo, no para anticipar movimientos intradía.

Si EURUSD = 1,20, un activo de 120 USD equivale a 100 EUR. No confundas USD por EUR con EUR por USD. No se rellenan precios ausentes con ceros ni se usa backward fill.

<div style="background-color:rgba(240,140,0,0.14); border-left:6px solid #f08c00; padding:12px 16px; border-radius:6px;">

<b style="color:#f08c00;">🔎 FASE DE VALIDACIÓN · ¿el calendario y la divisa son coherentes?</b>

`prepare_weekly` se detiene con un error si:

- Hay fechas duplicadas, o precios o tipos de cambio ≤ 0 o infinitos.
- No hay ninguna sesión en que coincidan todas las series.
- Quedan menos de 53 semanas, o falta alguna semana interior: **no se rellena** (ver E4).

Usa solo días de lunes a viernes con dato en **todas** las series, y convierte a EUR dividiendo por EURUSD (USD por EUR).

</div>

In [ ]:
print(inspect.getsource(prepare_weekly))

weekly_usd, weekly_eur, common_rows = prepare_weekly(raw_prices, START, END)
returns = weekly_eur.pct_change(fill_method=None).iloc[1:]
equity = weekly_eur.div(weekly_eur.iloc[0])*INITIAL_CAPITAL
print(f'Sesiones comunes: {common_rows} | Semanas: {len(weekly_eur)} | {weekly_eur.index.min().date()} a {weekly_eur.index.max().date()}')
display(equity.head(4).round(2))

## 4. Métricas comparables
CAGR usa años de calendario transcurridos; la volatilidad usa la desviación estándar muestral semanal multiplicada por $\sqrt{52}$. Esta anualización supone aproximadamente estabilidad temporal y poca dependencia. Drawdown es $E_t/\max_{s\le t} E_s -1$, incluyendo el capital inicial. El drag aquí se define como media aritmética menos geométrica semanal, ambas anualizadas linealmente por 52 para facilitar la comparación (no es la diferencia entre dos CAGR).

In [ ]:
print(inspect.getsource(performance))

metrics = performance(weekly_eur)
display(metrics.round(2))
fig, ax = plt.subplots(figsize=(10,4.5))
for asset in ASSETS: ax.plot(equity.index, equity[asset], label=asset)
ax.set(title=f'Capital acumulado en EUR · {DATA_MODE}', ylabel='EUR (base 1.000)', xlabel='Semana'); ax.legend(ncol=3); plt.show()

**Interpretación:** el activo que acaba con mayor capital no tiene por qué ser el que ofrece mejor relación rentabilidad/riesgo. Compara su volatilidad y su caída máxima antes de sacar conclusiones.

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Simulación: el más volátil es el que más pierde</b>

| Activo | Retorno | CAGR | Volatilidad | Caída máxima | Drag |
|---|---|---|---|---|---|
| BTC | −75,0 % | −24,4 % | **53,2 %** | **86,7 %** | **13,98 pp** |
| Acciones | **+45,6 %** | **+7,9 %** | 17,4 % | 18,1 % | 1,51 pp |
| Bonos | −11,7 % | −2,5 % | 8,9 % | 31,8 % | 0,39 pp |
| Oro | +20,4 % | +3,8 % | 13,3 % | 16,0 % | 0,88 pp |
| Global | −22,5 % | −5,0 % | 15,8 % | 34,5 % | 1,24 pp |
| USD efectivo | −6,6 % | −1,4 % | 4,8 % | 14,8 % | 0,11 pp |

- **El drag de volatilidad:** el retorno medio aritmético de BTC supera al geométrico en casi 14 puntos al año. Con un 53 % de volatilidad, las caídas pesan mucho más que las subidas del mismo tamaño (como +50 % y −50 %, que dejan un −25 %).
- Los bonos, con poca volatilidad, sufren una caída máxima del 32 %: volatilidad baja no significa caída pequeña.
- Los parámetros son arbitrarios: estos resultados no dicen nada sobre los activos reales (ver E7).

</div>

In [ ]:
drawdowns = equity.div(equity.cummax())-1
fig, ax = plt.subplots(figsize=(10,4.5))
for asset in ASSETS: ax.plot(drawdowns.index,100*drawdowns[asset],label=asset)
ax.set(title=f'Drawdown desde máximos · {DATA_MODE}', ylabel='Drawdown (%)', xlabel='Semana'); ax.legend(ncol=3); plt.show()

## 5. Correlación y efecto de la divisa
La correlación se calcula sobre retornos, no sobre precios. Compara la correlación de las distintas clases de activos; no es constante ni implica causalidad. La siguiente lección estudiará la diversificación formalmente.

In [ ]:
corr = returns.corr()
fig, ax = plt.subplots(figsize=(7,5.3)); image = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(ASSETS)), ASSETS, rotation=30, ha='right'); ax.set_yticks(range(len(ASSETS)), ASSETS)
for i in range(len(ASSETS)):
    for j in range(len(ASSETS)):
        value = corr.iloc[i,j]
        ax.text(j,i,f'{value:.2f}',ha='center',va='center',color='white' if abs(value)>.6 else 'black')
ax.set(title=f'Correlación de retornos semanales en EUR · {DATA_MODE}'); ax.grid(False); fig.colorbar(image, ax=ax, label='Pearson'); fig.tight_layout(); plt.show()

Para un inversor en EUR, $1+r_{EUR}=(1+r_{USD})/(1+r_{EURUSD})$. Una apreciación del euro reduce el rendimiento en EUR de un activo denominado en USD, manteniendo todo lo demás constante.

In [ ]:
print(inspect.getsource(fx_decomposition))

reconstructed = fx_decomposition(weekly_usd)
fx_compare = pd.DataFrame({'USD': 100*(weekly_usd['Acciones']/weekly_usd['Acciones'].iloc[0]-1),
                           'EUR': 100*(weekly_eur['Acciones']/weekly_eur['Acciones'].iloc[0]-1)})
ax = fx_compare.plot(title=f'Retorno acumulado de acciones por moneda · {DATA_MODE}')
ax.set(ylabel='Rentabilidad acumulada (%)',xlabel='Semana'); plt.show()
print(f'Retorno total Acciones en USD: {fx_compare.USD.iloc[-1]:.2f}% | en EUR: {fx_compare.EUR.iloc[-1]:.2f}%')
print(f"EURUSD en el periodo: {100*(weekly_usd.EURUSD.iloc[-1]/weekly_usd.EURUSD.iloc[0]-1):+.2f}%")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · La divisa cambia el resultado</b>

- En la simulación, las acciones suben un **55,9 % en USD** pero solo un **45,6 % en EUR**, porque el euro se aprecia un 7,1 % frente al dólar: (1 + 0,559) / (1 + 0,071) − 1 = 0,456.
- La reconstrucción (1 + r_USD) / (1 + r_EURUSD) − 1 coincide exactamente con los retornos en EUR, semana a semana.
- En la simulación, BTC, acciones y el ETF global se mueven juntos (correlaciones de 0,58 a 0,85, por el factor de mercado común); los bonos tienen correlación negativa con las acciones (−0,17).

</div>

## 6. Ejemplo de retorno total y rentabilidad real
El precio por sí solo omite dividendos. En un periodo, con dividendo pagado al final: retorno total = (precio final + dividendo) / precio inicial - 1. Para varios periodos se compone la serie. Para bonos, aplica el mismo principio a cupones si corresponde. No sumes dividendos a un cierre ya ajustado. La inflación debe corresponder al mismo horizonte y moneda.

In [ ]:
price_start, price_end, dividend = 100, 98, 4
price_return = price_end/price_start-1
total_ret = total_return(price_start, price_end, dividend)
real_return = (1+.08)/(1+.05)-1
print(f'Retorno por precio: {price_return:.2%} | retorno total: {total_ret:.2%}')
print(f'Retorno real (8% nominal, inflación 5%): {real_return:.2%}')

## 7. Comprobaciones automáticas
Estas pruebas validan identidades matemáticas y el tratamiento del calendario. No validan la exactitud del proveedor de mercado.

In [ ]:
# 🔎 FASE DE VALIDACIÓN: capitalización, retorno total, calendario y reconstrucción por divisa
assert np.isclose(future_value(1000,.1,2),1210)
assert np.isclose(present_value(1210,.1,2),1000)
assert np.isclose(compound_return([.5,-.5]),-.25)
assert np.isclose(real_return,.02857142857)
assert np.isclose(total_ret,.02)
assert len(returns)==len(equity)-1
assert np.allclose(equity.iloc[0],INITIAL_CAPITAL)
assert np.allclose((1+returns).prod(),equity.iloc[-1]/INITIAL_CAPITAL)
assert np.allclose(reconstructed,returns[reconstructed.columns])
assert (metrics.max_drawdown_pct>=0).all()
assert (metrics.drag_pp>=-1e-10).all()
assert (weekly_eur.index.to_series().diff().dropna()==pd.Timedelta(days=7)).all()
print('12 comprobaciones superadas')
if EXPORT:
    out=Path('quantlab_exports'); out.mkdir(exist_ok=True)
    raw_prices.to_csv(out/f'p14_prices_{DATA_MODE}.csv',index_label='date')
    weekly_eur.to_csv(out/f'p14_weekly_eur_{DATA_MODE}.csv',index_label='date')
    returns.to_csv(out/f'p14_returns_{DATA_MODE}.csv',index_label='date')
    metrics.to_csv(out/f'p14_metrics_{DATA_MODE}.csv',index_label='asset')

## 8. Ejercicios
1. **Capitalización:** implementa tu propia función con un bucle. Calcula el capital final para retornos +20%, −10%, +15%, −5%. Control: 1.179,90 EUR sobre 1.000 EUR.
2. **Tasas equivalentes:** calcula el tipo mensual equivalente a un tipo efectivo anual del 12%. ¿Por qué no es exactamente 1%? Control: aproximadamente 0,9489% mensual.
3. **CAGR:** compara 1.000 → 2.000 EUR en 3, 10 y 20 años. Explica la diferencia entre promedio aritmético y CAGR.
4. **Calendario:** elimina todas las observaciones de una semana interior en una copia. Comprueba que el validador rechaza el hueco y explica por qué rellenarlo sería peligroso.
5. **Divisa:** un activo sube 10% en USD mientras EURUSD sube 5%. Calcula el rendimiento en EUR sin usar las funciones de referencia. Control: 4,7619%.
6. **Retorno total:** construye un ejemplo de tres años de precios y dividendos. Compara reinversión frente a retorno de precio. No sumes dividendos sobre cierres ajustados.
7. **Análisis multiactivo real:** activa `live` o importa tu CSV. Identifica periodo, moneda, clase de activo, fuente, CAGR, volatilidad, drawdown y correlaciones. No declares vencedor solo por rentabilidad histórica.
8. **Riesgo y estabilidad:** divide la serie semanal cronológicamente en dos mitades, rebase cada subperiodo a 1.000 EUR incluyendo su punto inicial, y compara rankings y correlaciones. ¿Se mantienen? Define antes el criterio de comparación.

### E1–E3 · Capitalización, tasas equivalentes y CAGR

In [ ]:
print(inspect.getsource(manual_compound))
print(inspect.getsource(equivalent_rate))

e1 = manual_compound(1000, [.20, -.10, .15, -.05])
assert np.isclose(e1, 1179.90)
print(f"E1 · Capital final: {e1:,.2f} EUR (media aritmética de los retornos: {np.mean([.2,-.1,.15,-.05]):.2%})")
monthly = equivalent_rate(.12, 12)
print(f"E2 · Tipo mensual equivalente al 12 % anual: {monthly:.4%} | 12 meses al 1 %: {1.01**12-1:.2%} anual")
display(pd.DataFrame({'años': [3, 10, 20], 'CAGR': [cagr(1000, 2000, y) for y in (3, 10, 20)],
                      'media aritmética simple (100 % / años)': [1/y for y in (3, 10, 20)]}).set_index('años').style.format('{:.2%}'))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E1–E3 · Componer, no sumar ni dividir</b>

- **E1:** 1.000 × 1,20 × 0,90 × 1,15 × 0,95 = **1.179,90 €**. La media aritmética de los cuatro retornos es +5 %, pero el resultado no es "+5 % cuatro veces" (1.215,51 €): las caídas se aplican sobre un capital ya cambiado.
- **E2:** (1,12)^{1/12} − 1 = **0,9489 %** mensual. No es el 1 % porque los intereses mensuales también generan intereses: 12 meses al 1 % dan un **12,68 %** anual, no un 12 %.
- **E3:** duplicar el capital en 3, 10 y 20 años supone un CAGR del **25,99 %**, **7,18 %** y **3,53 %**. El "promedio aritmético" (100 % / años = 33,3 %, 10 % y 5 %) siempre es mayor porque ignora la composición: para duplicar en 10 años basta un 7,18 % compuesto, no un 10 % anual. Cuanto más largo el plazo, mayor la diferencia relativa.

</div>

### E4–E6 · Calendario, divisa y dividendos

In [ ]:
gap_week = raw_prices.index[(raw_prices.index >= '2023-03-06') & (raw_prices.index <= '2023-03-10')]
try:
    prepare_weekly(raw_prices.drop(index=gap_week), START, END)
except ValueError as exc:
    print(f"E4 · Semana del 6 al 10 de marzo de 2023 eliminada → ValueError: {exc}")
else:
    raise AssertionError('El validador no rechazó el hueco')

usd_up, eurusd_up = .10, .05
e5 = (1 + usd_up) / (1 + eurusd_up) - 1
assert np.isclose(e5, eur_return(usd_up, eurusd_up)) and np.isclose(e5, .047619, atol=1e-6)
print(f"E5 · Activo +10 % en USD con EURUSD +5 % → {e5:.4%} en EUR")

print(inspect.getsource(dividend_paths))
display(dividend_paths([100, 105, 98, 110], [3, 3, 3]).round(3))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E4–E6 · Huecos, divisa y dividendos</b>

- **E4:** al borrar la semana del 6 al 10 de marzo de 2023, el validador se detiene. **Rellenar el hueco sería peligroso:** con el último precio, la semana tendría un retorno de exactamente 0 y la siguiente acumularía dos semanas de movimiento; con una interpolación, se inventarían precios que nadie negoció. En los dos casos se infravalora la volatilidad y se distorsionan las correlaciones.
- **E5:** 1,10 / 1,05 − 1 = **4,7619 %**. La subida del euro "se come" casi la mitad de la ganancia en dólares: con cada USD se compran menos euros. No es 10 − 5 = 5 % porque los efectos se componen.
- **E6:** precios 100 → 105 → 98 → 110 con 3 € de dividendo al año:

| Año | Retorno de precio | Total cobrando los dividendos | Total reinvirtiéndolos |
|---|---|---|---|
| 1 | +5,0 % | +8,0 % | +8,0 % |
| 2 | −2,0 % | +4,0 % | +3,9 % |
| 3 | **+10,0 %** | **+19,0 %** | **+19,8 %** |

  - El retorno de precio omite 9 puntos de los 19. Reinvertir gana 0,8 puntos más al final, aunque en el año 2 queda ligeramente por detrás: el dividendo del año 1 se reinvirtió a 105 y el precio cayó a 98.
  - Con cierres **ajustados** (como los de la sección 2), los dividendos ya están dentro: sumarlos otra vez contaría el dividendo dos veces.

</div>

### E7 · Análisis multiactivo con datos reales
Cambia `LIVE_E7` a `True` para descargar los cierres ajustados de Yahoo Finance (requiere `yfinance`, incluido en `requirements.txt`). Si la descarga falla, la celda se detiene con el error.

In [ ]:
LIVE_E7 = False

if LIVE_E7:
    live_raw = download_prices(START, END)
    live_usd, live_eur, live_common = prepare_weekly(live_raw, START, END)
    print(f"Yahoo Finance · descarga {pd.Timestamp.now(tz='UTC'):%Y-%m-%d %H:%M} UTC | sesiones comunes: {live_common} | "
          f"semanas: {len(live_eur)} | {live_eur.index.min().date()} a {live_eur.index.max().date()} | moneda: EUR")
    display(performance(live_eur).round(2))
    display(live_eur.pct_change(fill_method=None).iloc[1:].corr().round(2))
    print(f"Acciones (SPY): {100*(live_usd.Acciones.iloc[-1]/live_usd.Acciones.iloc[0]-1):.2f}% en USD | "
          f"{100*(live_eur.Acciones.iloc[-1]/live_eur.Acciones.iloc[0]-1):.2f}% en EUR | EURUSD {100*(live_usd.EURUSD.iloc[-1]/live_usd.EURUSD.iloc[0]-1):+.2f}%")
else:
    print("LIVE_E7 = False: no se descarga nada. Resultados documentados en la conclusión de abajo.")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E7 · Datos reales (Yahoo Finance, descarga del 19-11-2026)</b>

**Periodo:** semanas del 15-01-2021 al 26-12-2025 (259 semanas, 1.254 sesiones comunes). **Moneda:** EUR (precios en USD convertidos con EURUSD=X). **Fuente:** Yahoo Finance vía `yfinance`, cierres ajustados (incluyen dividendos reinvertidos de los ETF). **Clases:** cripto (BTC-USD), renta variable EE. UU. (SPY), bonos del Tesoro a 7–10 años (IEF), oro (GLD), renta variable global (VT) y dólar en efectivo.

| Activo | Retorno | CAGR | Volatilidad | Caída máxima | Drag |
|---|---|---|---|---|---|
| BTC | +144,4 % | +19,8 % | **57,4 %** | **72,2 %** | 16,34 pp |
| Acciones (SPY) | +102,9 % | +15,4 % | 15,8 % | 21,4 % | 1,25 pp |
| Bonos (IEF) | −4,4 % | −0,9 % | 9,1 % | 15,4 % | 0,42 pp |
| Oro (GLD) | **+151,1 %** | **+20,5 %** | 14,0 % | **9,0 %** | 0,97 pp |
| Global (VT) | +72,6 % | +11,7 % | 14,4 % | 18,7 % | 1,03 pp |
| USD efectivo | +3,1 % | +0,6 % | 7,5 % | 17,4 % | 0,28 pp |

**Correlaciones semanales en EUR:** SPY–VT 0,97 (casi el mismo activo); BTC con acciones, solo 0,25; bonos–USD 0,63 (los dos reaccionan al dólar y a los tipos de EE. UU.); oro, por debajo de 0,25 con todo.

**Efecto de la divisa:** SPY subió un 96,8 % en USD y un **102,9 % en EUR**, porque el euro se depreció un 3,0 % frente al dólar.

**Sin declarar vencedor:** el oro tuvo el mayor CAGR (20,5 %) con una caída máxima de solo un 9 %, pero en un periodo excepcional para el oro. BTC rindió parecido con cuatro veces su volatilidad y una caída del 72 %. Los bonos perdieron en el ciclo de subidas de tipos de 2022. Son cinco años concretos: no dicen qué activo rendirá más en el futuro.

</div>

### E8 · Estabilidad: dos mitades

In [ ]:
print(inspect.getsource(split_halves))

# Criterio fijado antes de mirar: comparar el ORDEN por CAGR y la correlación BTC–Acciones y Acciones–Bonos.
halves = split_halves(weekly_eur, INITIAL_CAPITAL)
for name, h in halves.items():
    print(f"{name}: {h['start'].date()} → {h['end'].date()}")
display(pd.DataFrame({name: h['metrics'].cagr_pct for name, h in halves.items()}).round(2).sort_values('primera', ascending=False))
display(pd.DataFrame({name: {'BTC–Acciones': h['corr'].loc['BTC', 'Acciones'], 'Acciones–Bonos': h['corr'].loc['Acciones', 'Bonos']}
                      for name, h in halves.items()}).round(2))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E8 · Ni el ranking ni las correlaciones son estables</b>

**Criterio fijado antes de mirar:** el orden de los activos por CAGR y dos correlaciones (BTC–Acciones y Acciones–Bonos). Cada mitad se rebasa a 1.000 € desde su propio punto inicial (la segunda empieza el 07-07-2023, donde termina la primera).

| CAGR | Simulación: 1.ª mitad | 2.ª mitad | Datos reales: 1.ª mitad | 2.ª mitad |
|---|---|---|---|---|
| BTC | −7,9 % | **−38,0 %** | −3,4 % | **+48,5 %** |
| Acciones | +7,1 % | +8,6 % | +13,0 % | +17,9 % |
| Bonos | −9,2 % | +4,8 % | −3,0 % | +1,2 % |
| Oro | **+10,7 %** | −2,7 % | +6,4 % | +36,4 % |
| Global | −5,6 % | −4,4 % | +7,4 % | +16,1 % |

- **El ranking cambia por completo.** En los datos reales, BTC pasa de ser el peor a ser el mejor, y el oro, de discreto a excepcional. En la simulación, el oro pasa de primero a perder.
- **Las correlaciones se mueven menos, pero se mueven:** en los datos reales, BTC–Acciones va de 0,28 a 0,21 y Acciones–Bonos, de 0,10 a 0,23. En la simulación se mantienen más estables (BTC–Acciones ≈ 0,58), porque el modelo tiene correlaciones fijas.
- **Lección:** con 2,5 años por mitad, el "mejor activo" depende del periodo elegido. Una conclusión basada en una sola ventana histórica no es robusta.

</div>

## 9. Conclusiones

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIONES DE LA PRÁCTICA P14</b>

**¿Qué activos han crecido más y qué riesgo han soportado?** En los datos reales de 2021–2025, el oro (+20,5 % anual, caída máxima del 9 %) y BTC (+19,8 % anual, pero con una volatilidad del 57 % y una caída del 72 %). Las acciones de EE. UU. rindieron un 15,4 % anual con una caída del 21 %. Los bonos perdieron. Rentabilidad y riesgo hay que leerlos juntos: BTC y el oro rindieron parecido con riesgos muy distintos.

**¿Cómo ha influido la moneda de referencia?** Para un inversor en euros, un activo en dólares gana o pierde además la variación del EURUSD. En el periodo real, el euro se depreció un 3 %, y SPY rindió 6 puntos más en EUR que en USD. En la simulación ocurrió lo contrario: el euro se apreció un 7 % y restó 10 puntos.

**¿Qué diferencias existen entre retorno de precio, total y real?** El de precio solo mira el precio; el total suma los dividendos o cupones (reinvertidos, si se componen); el real descuenta la inflación del mismo periodo y moneda. Con los cierres ajustados de los ETF, el retorno ya es total: no hay que sumar los dividendos otra vez.

**¿Qué cambiaría para una investigación de alta frecuencia?** Los cierres diarios de mercados distintos no son simultáneos (BTC cotiza 24/7). Harían falta timestamps exactos, precios del mismo instante en todos los mercados, tipos de cambio intradía, costes de ejecución y datos point-in-time, en lugar de cierres ajustados y revisados.

Los datos sintéticos no permiten concluir qué activos serán más rentables en el futuro, y los reales tampoco: cinco años, divididos en dos, ya cambian el ranking.

📄 Resumen en `docs/conclusiones/P14_multi_asset_analysis.md`

</div>

**Siguiente:** 1.29 (diversificación y riesgos sistemático/idiosincrático) + P15 (regímenes macroeconómicos).